# ROC-Based Predictive Analysis: 30-Day Price Movement Classification
## XGBoost Classifier

This notebook implements a supervised machine learning workflow to predict 30-day NVDA price movements using XGBoost.

**Data source:** `model_data.csv` — pre-processed by `DataPipeline.ipynb`  
- Cleaning, feature engineering, multicollinearity reduction, and target variable creation are all handled upstream.
- This notebook loads the ready-to-use dataset and focuses entirely on model training and evaluation.

**Target:** `Price_Direction_30d`
- `1` = Positive 30-day forward ROC (price goes up)
- `0` = Non-positive 30-day forward ROC (price flat or down)


## Initial imports and basic settings

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:.4f}' if abs(x) < 10000 else f'{x:.0f}')
sns.set_style("whitegrid")

## Load data from shared pipeline

Cleaning, feature selection, and target engineering are already done in `DataPipeline.ipynb`.  
No additional preprocessing needed here.


In [ ]:
df = pd.read_csv('../model_data.csv', parse_dates=['Date'])
df = df.sort_values('Date').reset_index(drop=True)

print(f"Shape      : {df.shape}")
print(f"Date range : {df['Date'].min().date()} → {df['Date'].max().date()}")
print(f"NaN values : {df.isna().sum().sum()}")
df.head()

In [ ]:
print("Features in dataset:")
feature_cols = [c for c in df.columns if c not in ['Date', 'Price_Direction_30d']]
for f in feature_cols:
    print(f"  {f}")
print(f"\nTotal features: {len(feature_cols)}")

## ROC descriptive statistics

In [ ]:
roc = df['ROC_roll5_mean'].dropna()
print("=" * 80)
print("ROC_roll5_mean STATISTICAL ANALYSIS")
print("=" * 80)
print(f"\nTotal observations: {len(roc)}")
print(f"\n{'Metric':<25} {'Value':<15}")
print("-" * 40)
for label, val in [
    ('Mean',     roc.mean()),
    ('Median',   roc.median()),
    ('Std Dev',  roc.std()),
    ('Skewness', roc.skew()),
    ('Kurtosis', roc.kurtosis()),
    ('Min',      roc.min()),
    ('Max',      roc.max()),
    ('IQR',      roc.quantile(0.75) - roc.quantile(0.25)),
]:
    print(f"  {label:<23} {val:<15.4f}")


## XGBoost Classifier — Model Training & Evaluation

In [ ]:
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix,
                              classification_report, roc_curve, auc)


FEATURES = ['MACD_Signal', 'BB_Width', 'RSI_lag1', 'ATR_Pct', 'SMA20_Dist', 'Stoch_K', 'ROC_roll5_mean', 'Volume_Ratio', 'CCI']
TARGET   = 'Price_Direction_30d'

X = df[FEATURES]
y = df[TARGET]

print("=" * 80)
print("DATA PREPARATION FOR XGBOOST")
print("=" * 80)
print(f"\nFeatures shape: {X.shape}")
print(f"Target shape  : {y.shape}")
print(f"Features used : {len(FEATURES)}")
print(f"Total samples : {len(df)}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

print(f"\nTrain/Test Split:")
print(f"Training set size: {X_train.shape[0]} samples")
print(f"Testing set size : {X_test.shape[0]} samples")
print(f"Train target distribution:\n{y_train.value_counts(normalize=True) * 100}")
print(f"\nTest target distribution:\n{y_test.value_counts(normalize=True) * 100}")


In [ ]:
from sklearn.model_selection import TimeSeriesSplit

# Temporal CV — respects chronological order, no future leakage into past folds
tscv = TimeSeriesSplit(n_splits=5)

neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
spw = neg / pos   # class imbalance ratio

# ── Heavily regularised search space — combats severe overfitting on noisy
#    financial features. Prior run had train acc 0.88 vs test 0.53 (gap 0.35).
#    The new grid favours shallow trees, slow learning, strong L1/L2,
#    high min_child_weight, and aggressive sub-sampling. ──────────────────────
param_dist = {
    'n_estimators':     [100, 150, 200, 300],
    'max_depth':        [2, 3],                 # very shallow → cannot memorise
    'learning_rate':    [0.01, 0.02, 0.03],     # slow learning
    'subsample':        [0.5, 0.6, 0.7],        # row sub-sampling
    'colsample_bytree': [0.4, 0.5, 0.6],        # column sub-sampling
    'min_child_weight': [10, 20, 30, 50],       # higher → conservative splits
    'reg_alpha':        [0.5, 1.0, 2.0, 5.0],   # strong L1
    'reg_lambda':       [2.0, 5.0, 10.0],       # strong L2
    'scale_pos_weight': [1, spw],
    'gamma':            [0.1, 0.3, 0.5, 1.0],   # min split-loss reduction
}

xgb_base = XGBClassifier(
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1,
    tree_method='hist',
)

xgb_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_dist,
    n_iter=60,
    scoring='roc_auc',
    cv=tscv,
    verbose=1,
    random_state=42,
    n_jobs=-1,
    refit=True,
)

xgb_search.fit(X_train, y_train)

# ── Final refit with early stopping on a held-out tail of the training set ──
#    Carve the last 15% of training (chronological) as validation so the final
#    model's n_estimators is tuned by early stopping rather than risking
#    over-training during the post-search refit.
val_size  = int(len(X_train) * 0.15)
X_tr_es   = X_train.iloc[:-val_size]
y_tr_es   = y_train.iloc[:-val_size]
X_val_es  = X_train.iloc[-val_size:]
y_val_es  = y_train.iloc[-val_size:]

best_params = xgb_search.best_params_.copy()
# Allow more rounds; early stopping will halt before overfitting kicks in.
best_params['n_estimators'] = 1000

best_xgb = XGBClassifier(
    **best_params,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1,
    tree_method='hist',
    early_stopping_rounds=30,
)
best_xgb.fit(
    X_tr_es, y_tr_es,
    eval_set=[(X_val_es, y_val_es)],
    verbose=False,
)

print("\nBest Parameters (Model 1 — All Features, regularised):")
for k, v in xgb_search.best_params_.items():
    print(f"  {k:<20} {v}")
print(f"\nBest CV ROC-AUC                : {xgb_search.best_score_:.4f}")
print(f"Final n_estimators (early-stop): {best_xgb.best_iteration + 1}")

y_pred_train = best_xgb.predict(X_train)
y_pred_test  = best_xgb.predict(X_test)
y_pred_proba = best_xgb.predict_proba(X_test)[:, 1]

print("\nTest Performance (Model 1 — All 9 Features):")
print(classification_report(y_test, y_pred_test))

In [ ]:
def evaluate_model(model, X_train, y_train, X_test, y_test, label, features,
                   cv_score=None):
    """Return a metrics dict and print a formatted summary."""
    y_pred_tr = model.predict(X_train[features])
    y_pred_te = model.predict(X_test[features])
    y_proba_te = model.predict_proba(X_test[features])[:, 1]

    metrics = {
        'model':          label,
        'n_features':     len(features),
        'features':       list(features),
        'cv_roc_auc':     cv_score,
        'train_accuracy': accuracy_score(y_train, y_pred_tr),
        'train_f1':       f1_score(y_train, y_pred_tr),
        'test_accuracy':  accuracy_score(y_test, y_pred_te),
        'test_precision': precision_score(y_test, y_pred_te, zero_division=0),
        'test_recall':    recall_score(y_test, y_pred_te, zero_division=0),
        'test_f1':        f1_score(y_test, y_pred_te, zero_division=0),
        'test_roc_auc':   roc_auc_score(y_test, y_proba_te),
        'overfit_gap':    accuracy_score(y_train, y_pred_tr)
                          - accuracy_score(y_test, y_pred_te),
    }
    print(f"\n{'='*80}\n{label.upper()}\n{'='*80}")
    print(f"  Features used  : {len(features)}  →  {features}")
    if cv_score is not None:
        print(f"  CV ROC-AUC     : {cv_score:.4f}")
    print(f"  Train accuracy : {metrics['train_accuracy']:.4f}")
    print(f"  Test accuracy  : {metrics['test_accuracy']:.4f}")
    print(f"  Test F1        : {metrics['test_f1']:.4f}")
    print(f"  Test ROC-AUC   : {metrics['test_roc_auc']:.4f}")
    print(f"  Overfit gap    : {metrics['overfit_gap']:+.4f}  "
          f"(train acc − test acc — lower is better)")
    return metrics


# Registry that every subsequent model will append to.
results = {}
results['Model 1 — All Features'] = evaluate_model(
    best_xgb, X_train, y_train, X_test, y_test,
    label='Model 1 — All Features',
    features=FEATURES,
    cv_score=xgb_search.best_score_,
)

print("\n" + "=" * 80)
print("DETAILED CLASSIFICATION REPORT — Model 1 (Test Set)")
print("=" * 80)
print(classification_report(y_test, y_pred_test, target_names=['Negative', 'Positive']))

In [ ]:
print("=" * 80)
print("FEATURE IMPORTANCE")
print("=" * 80)

feature_importance = pd.DataFrame({
    'Feature':    FEATURES,
    'Importance': best_xgb.feature_importances_
}).sort_values('Importance', ascending=False)

print("\nTop 10 Most Important Features:")
print(feature_importance.head(10).to_string(index=False))

fig, axes = plt.subplots(2, 1, figsize=(12, 10))

axes[0].barh(feature_importance['Feature'], feature_importance['Importance'], color='steelblue')
axes[0].set_xlabel('Importance Score', fontsize=11)
axes[0].set_title('Feature Importance — XGBoost', fontweight='bold', fontsize=12)
axes[0].invert_yaxis()

cumsum_importance = feature_importance['Importance'].cumsum()
axes[1].plot(range(1, len(cumsum_importance) + 1), cumsum_importance.values,
             marker='o', linestyle='-', linewidth=2)
axes[1].axhline(0.95, color='red', linestyle='--', label='95% Cumulative Importance')
axes[1].set_xlabel('Number of Features', fontsize=11)
axes[1].set_ylabel('Cumulative Importance', fontsize=11)
axes[1].set_title('Cumulative Feature Importance', fontweight='bold', fontsize=12)
axes[1].grid(alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()

n_features_95 = (cumsum_importance >= 0.95).argmax() + 1
print(f"\nNumber of features for 95% cumulative importance: {n_features_95}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm = confusion_matrix(y_test, y_pred_test)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0], cbar=False,
            xticklabels=['Negative', 'Positive'],
            yticklabels=['Negative', 'Positive'])
axes[0].set_ylabel('True Label', fontsize=11)
axes[0].set_xlabel('Predicted Label', fontsize=11)
axes[0].set_title('Confusion Matrix (Test Set)', fontweight='bold', fontsize=12)

tn, fp, fn, tp = cm.ravel()
axes[0].text(0.5, -0.25, f'TN={tn} FP={fp}\nFN={fn} TP={tp}',
             transform=axes[0].transAxes, ha='center', fontsize=10, family='monospace')

fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
roc_auc_val = auc(fpr, tpr)

axes[1].plot(fpr, tpr, color='darkorange', lw=2.5,
             label=f'ROC curve (AUC = {roc_auc_val:.4f})')
axes[1].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier')
axes[1].set_xlim([0.0, 1.0])
axes[1].set_ylim([0.0, 1.05])
axes[1].set_xlabel('False Positive Rate', fontsize=11)
axes[1].set_ylabel('True Positive Rate', fontsize=11)
axes[1].set_title('ROC Curve (Test Set)', fontweight='bold', fontsize=12)
axes[1].legend(loc="lower right", fontsize=10)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# ── Model 1 summary (uses the regularised model + results registry) ──────────
print("=" * 80)
print("MODEL 1 SUMMARY & PREDICTIONS")
print("=" * 80)

m1 = results['Model 1 — All Features']

predictions_df = pd.DataFrame({
    'Actual':      y_test.values,
    'Predicted':   y_pred_test,
    'Probability': y_pred_proba,
    'Correct':     y_test.values == y_pred_test,
})

print(f"\nTotal Test Predictions: {len(predictions_df)}")
print(f"Correct Predictions   : {predictions_df['Correct'].sum()}")
print(f"Incorrect Predictions : {(~predictions_df['Correct']).sum()}")

print("\nRecent Predictions (Last 10 samples):")
print(predictions_df.tail(10).to_string())

print("\n" + "=" * 80)
print("PREDICTION CONFIDENCE ANALYSIS")
print("=" * 80)
high_conf = (predictions_df['Probability'] >= 0.7) | (predictions_df['Probability'] <= 0.3)
print(f"\nHigh Confidence (prob ≥ 0.7 or ≤ 0.3): "
      f"{high_conf.sum()} ({high_conf.sum()/len(predictions_df)*100:.1f}%)")
print(f"Low  Confidence (0.3 < prob < 0.7)   : "
      f"{(~high_conf).sum()} ({(~high_conf).sum()/len(predictions_df)*100:.1f}%)")

print("\n" + "=" * 80)
print("KEY TAKEAWAYS — MODEL 1")
print("=" * 80)
print(f"• Test accuracy : {m1['test_accuracy']:.2%}")
print(f"• Test ROC-AUC  : {m1['test_roc_auc']:.4f}")
print(f"• Test F1       : {m1['test_f1']:.4f}")
print(f"• Overfit gap   : {m1['overfit_gap']:+.4f}  "
      f"(was 0.3478 in the previous unregularised run)")
print(f"• Top predictor : {feature_importance.iloc[0]['Feature']} "
      f"(importance: {feature_importance.iloc[0]['Importance']:.4f})")
print(f"• Class balance : {y_test.value_counts(normalize=True)[1]:.1%} positive in test")
print("=" * 80)

## Model 2 — Feature-Eliminated XGBoost (95% Cumulative Importance)

Using Model 1's feature importance scores to select only the features that together explain **≥ 95%** of the model's predictive power. The remainder are dropped.  
Model 2 is then retrained on this reduced feature set using the same temporal CV and search space.

In [ ]:
# ── Feature selection: retain features up to 95% cumulative importance ───────
print("=" * 80)
print("FEATURE IMPORTANCE — MODEL 1 (All 9 Features)")
print("=" * 80)

feature_importance_m1 = pd.DataFrame({
    'Feature':    FEATURES,
    'Importance': best_xgb.feature_importances_,
}).sort_values('Importance', ascending=False).reset_index(drop=True)
feature_importance_m1['Cumulative'] = feature_importance_m1['Importance'].cumsum()

print(f"\n{'Rank':<6} {'Feature':<20} {'Importance':<15} {'Cumulative'}")
print("-" * 58)
for i, row in feature_importance_m1.iterrows():
    prev_cum = feature_importance_m1.iloc[i - 1]['Cumulative'] if i > 0 else 0
    marker = "  ← 95% cut" if row['Cumulative'] >= 0.95 and prev_cum < 0.95 else ""
    print(f"  {i+1:<4} {row['Feature']:<20} {row['Importance']:.4f}"
          f"{'':>8} {row['Cumulative']:.4f}{marker}")

# Minimum number of features whose cumulative importance reaches 95%.
n_features_95 = int((feature_importance_m1['Cumulative'] < 0.95).sum()) + 1
n_features_95 = min(n_features_95, len(FEATURES))
FEATURES_M2   = feature_importance_m1.iloc[:n_features_95]['Feature'].tolist()
dropped_feats = feature_importance_m1.iloc[n_features_95:]['Feature'].tolist()

print(f"\n→ Retaining {len(FEATURES_M2)} features — cumulative importance: "
      f"{feature_importance_m1.iloc[n_features_95-1]['Cumulative']:.2%}")
print(f"→ Dropped  : {dropped_feats if dropped_feats else 'none'}")

# ── Visualise ───────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['steelblue' if f in FEATURES_M2 else '#d62728'
          for f in feature_importance_m1['Feature']]
axes[0].barh(feature_importance_m1['Feature'], feature_importance_m1['Importance'],
             color=colors, edgecolor='black', linewidth=0.5)
axes[0].invert_yaxis()
axes[0].set_xlabel('Importance Score', fontsize=11)
axes[0].set_title('Feature Importance — Model 1\n(blue = retained, red = dropped)',
                  fontweight='bold', fontsize=12)

axes[1].plot(range(1, len(feature_importance_m1) + 1),
             feature_importance_m1['Cumulative'],
             marker='o', linewidth=2, color='steelblue')
axes[1].axhline(0.95, color='red', linestyle='--', linewidth=1.5, label='95% threshold')
axes[1].axvline(n_features_95, color='green', linestyle='--', linewidth=1.5,
                label=f'Cut at feature #{n_features_95}')
axes[1].set_xlabel('Number of Features', fontsize=11)
axes[1].set_ylabel('Cumulative Importance', fontsize=11)
axes[1].set_title('Cumulative Feature Importance', fontweight='bold', fontsize=12)
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ── Retrain XGBoost on the reduced feature set with the same regularised
#    search space, then early-stop on the validation tail. ───────────────────
X_train_m2 = X_train[FEATURES_M2]
X_test_m2  = X_test[FEATURES_M2]

xgb_search_m2 = RandomizedSearchCV(
    estimator=XGBClassifier(eval_metric='logloss', random_state=42,
                            n_jobs=-1, tree_method='hist'),
    param_distributions=param_dist,
    n_iter=60,
    scoring='roc_auc',
    cv=tscv,
    verbose=1,
    random_state=42,
    n_jobs=-1,
    refit=True,
)
xgb_search_m2.fit(X_train_m2, y_train)

best_params_m2 = xgb_search_m2.best_params_.copy()
best_params_m2['n_estimators'] = 1000

best_xgb_m2 = XGBClassifier(
    **best_params_m2,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1,
    tree_method='hist',
    early_stopping_rounds=30,
)
best_xgb_m2.fit(
    X_train_m2.iloc[:-val_size], y_train.iloc[:-val_size],
    eval_set=[(X_train_m2.iloc[-val_size:], y_train.iloc[-val_size:])],
    verbose=False,
)

print("\nBest Parameters (Model 2 — 95% Importance Subset):")
for k, v in xgb_search_m2.best_params_.items():
    print(f"  {k:<20} {v}")
print(f"\nBest CV ROC-AUC                : {xgb_search_m2.best_score_:.4f}")
print(f"Final n_estimators (early-stop): {best_xgb_m2.best_iteration + 1}")

results['Model 2 — 95% Importance'] = evaluate_model(
    best_xgb_m2, X_train, y_train, X_test, y_test,
    label='Model 2 — 95% Importance',
    features=FEATURES_M2,
    cv_score=xgb_search_m2.best_score_,
)

## Model 3 — Recursive Feature Elimination with Time-Series CV (RFECV)

Uses `sklearn.feature_selection.RFECV` to recursively drop the weakest feature, scoring each subset by ROC-AUC under the same `TimeSeriesSplit` (no temporal leakage). The optimal feature count is chosen automatically. Model 3 is then retrained on that subset using the full regularised hyperparameter search.

In [ ]:
from sklearn.feature_selection import RFECV

# A modest-but-regularised XGB used as the RFE estimator. We *don't* run a
# full hyperparameter search inside RFECV (that would be O(folds × subsets ×
# search_iters) and very slow); instead we use a sensible default and then
# refit the final model with the full search on the chosen subset.
rfe_estimator = XGBClassifier(
    n_estimators=200,
    max_depth=3,
    learning_rate=0.03,
    subsample=0.7,
    colsample_bytree=0.6,
    min_child_weight=20,
    reg_alpha=1.0,
    reg_lambda=3.0,
    gamma=0.3,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1,
    tree_method='hist',
)

rfecv = RFECV(
    estimator=rfe_estimator,
    step=1,
    cv=tscv,
    scoring='roc_auc',
    min_features_to_select=2,
    n_jobs=-1,
)
rfecv.fit(X_train, y_train)

FEATURES_M3 = [f for f, keep in zip(FEATURES, rfecv.support_) if keep]
print(f"RFECV optimal feature count: {rfecv.n_features_}")
print(f"Selected features         : {FEATURES_M3}")
print(f"Eliminated features       : "
      f"{[f for f in FEATURES if f not in FEATURES_M3]}")

mean_scores = rfecv.cv_results_['mean_test_score']
plt.figure(figsize=(9, 4))
plt.plot(range(2, len(mean_scores) + 2), mean_scores,
         marker='o', color='steelblue', linewidth=2)
plt.axvline(rfecv.n_features_, color='green', linestyle='--',
            label=f'Optimal: {rfecv.n_features_} features')
plt.xlabel('Number of Features Selected')
plt.ylabel('Mean CV ROC-AUC')
plt.title('RFECV — Mean ROC-AUC vs. Feature Count', fontweight='bold')
plt.grid(alpha=0.3); plt.legend()
plt.tight_layout(); plt.show()

In [ ]:
# ── Retrain Model 3 with the full regularised search on the RFECV subset ────
X_train_m3 = X_train[FEATURES_M3]
X_test_m3  = X_test[FEATURES_M3]

xgb_search_m3 = RandomizedSearchCV(
    estimator=XGBClassifier(eval_metric='logloss', random_state=42,
                            n_jobs=-1, tree_method='hist'),
    param_distributions=param_dist,
    n_iter=60,
    scoring='roc_auc',
    cv=tscv,
    verbose=1,
    random_state=42,
    n_jobs=-1,
    refit=True,
)
xgb_search_m3.fit(X_train_m3, y_train)

best_params_m3 = xgb_search_m3.best_params_.copy()
best_params_m3['n_estimators'] = 1000

best_xgb_m3 = XGBClassifier(
    **best_params_m3,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1,
    tree_method='hist',
    early_stopping_rounds=30,
)
best_xgb_m3.fit(
    X_train_m3.iloc[:-val_size], y_train.iloc[:-val_size],
    eval_set=[(X_train_m3.iloc[-val_size:], y_train.iloc[-val_size:])],
    verbose=False,
)

print("\nBest Parameters (Model 3 — RFECV Subset):")
for k, v in xgb_search_m3.best_params_.items():
    print(f"  {k:<20} {v}")
print(f"\nBest CV ROC-AUC                : {xgb_search_m3.best_score_:.4f}")
print(f"Final n_estimators (early-stop): {best_xgb_m3.best_iteration + 1}")

results['Model 3 — RFECV'] = evaluate_model(
    best_xgb_m3, X_train, y_train, X_test, y_test,
    label='Model 3 — RFECV',
    features=FEATURES_M3,
    cv_score=xgb_search_m3.best_score_,
)

## Model 4 — Exhaustive Subset Search (Brute Force)

Evaluates **every** non-empty subset of the 9 features (2⁹ − 1 = 511 combinations) using a single fixed regularised XGBoost scored by mean `TimeSeriesSplit` CV ROC-AUC. The winning subset is then retrained with the full hyperparameter search.

This is the most thorough comparison possible — it guarantees we don't miss a strong feature combination that greedy RFE or importance-based pruning would skip.

In [ ]:
from itertools import combinations
from sklearn.model_selection import cross_val_score
import time

# A single, regularised XGB used to score each subset. Using one fixed model
# (not a full search) keeps the brute force tractable: 511 subsets × 5 folds
# = 2,555 fits — finishes in tens of seconds.
brute_estimator = XGBClassifier(
    n_estimators=200,
    max_depth=3,
    learning_rate=0.03,
    subsample=0.7,
    colsample_bytree=0.6,
    min_child_weight=20,
    reg_alpha=1.0,
    reg_lambda=3.0,
    gamma=0.3,
    eval_metric='logloss',
    random_state=42,
    n_jobs=1,           # parallelise across subsets, not within
    tree_method='hist',
)

n_combos = sum(1 for r in range(1, len(FEATURES) + 1)
               for _ in combinations(FEATURES, r))
print(f"Evaluating {n_combos} feature subsets (this may take a minute)…")

start = time.time()
subset_scores = []
for r in range(1, len(FEATURES) + 1):
    for subset in combinations(FEATURES, r):
        scores = cross_val_score(
            brute_estimator, X_train[list(subset)], y_train,
            cv=tscv, scoring='roc_auc', n_jobs=-1,
        )
        subset_scores.append({
            'features':   list(subset),
            'n_features': r,
            'cv_auc':     scores.mean(),
            'cv_std':     scores.std(),
        })
elapsed = time.time() - start
print(f"Done in {elapsed:.1f}s")

subset_df = (pd.DataFrame(subset_scores)
             .sort_values('cv_auc', ascending=False)
             .reset_index(drop=True))

print("\nTop 10 feature subsets by CV ROC-AUC:")
top10 = subset_df.head(10).copy()
top10['features'] = top10['features'].apply(lambda fs: ', '.join(fs))
print(top10[['n_features', 'cv_auc', 'cv_std', 'features']].to_string(index=False))

FEATURES_M4 = subset_df.iloc[0]['features']
best_brute_auc = subset_df.iloc[0]['cv_auc']
print(f"\n→ Brute-force winner: {len(FEATURES_M4)} features, "
      f"CV ROC-AUC = {best_brute_auc:.4f}")
print(f"→ Selected features : {FEATURES_M4}")

# Visualise: best CV AUC achievable at each feature-count cardinality
best_per_size = subset_df.loc[subset_df.groupby('n_features')['cv_auc'].idxmax()]
plt.figure(figsize=(9, 4))
plt.plot(best_per_size['n_features'], best_per_size['cv_auc'],
         marker='o', color='steelblue', linewidth=2)
plt.axvline(len(FEATURES_M4), color='green', linestyle='--',
            label=f'Optimal: {len(FEATURES_M4)} features')
plt.xlabel('Number of Features in Subset')
plt.ylabel('Best CV ROC-AUC at this size')
plt.title('Brute-Force Subset Search — Best Score per Cardinality',
          fontweight='bold')
plt.grid(alpha=0.3); plt.legend()
plt.tight_layout(); plt.show()

In [ ]:
# ── Retrain Model 4 with the full regularised search on the brute-force pick ──
X_train_m4 = X_train[FEATURES_M4]
X_test_m4  = X_test[FEATURES_M4]

xgb_search_m4 = RandomizedSearchCV(
    estimator=XGBClassifier(eval_metric='logloss', random_state=42,
                            n_jobs=-1, tree_method='hist'),
    param_distributions=param_dist,
    n_iter=60,
    scoring='roc_auc',
    cv=tscv,
    verbose=1,
    random_state=42,
    n_jobs=-1,
    refit=True,
)
xgb_search_m4.fit(X_train_m4, y_train)

best_params_m4 = xgb_search_m4.best_params_.copy()
best_params_m4['n_estimators'] = 1000

best_xgb_m4 = XGBClassifier(
    **best_params_m4,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1,
    tree_method='hist',
    early_stopping_rounds=30,
)
best_xgb_m4.fit(
    X_train_m4.iloc[:-val_size], y_train.iloc[:-val_size],
    eval_set=[(X_train_m4.iloc[-val_size:], y_train.iloc[-val_size:])],
    verbose=False,
)

print("\nBest Parameters (Model 4 — Brute-Force Subset):")
for k, v in xgb_search_m4.best_params_.items():
    print(f"  {k:<20} {v}")
print(f"\nBest CV ROC-AUC                : {xgb_search_m4.best_score_:.4f}")
print(f"Final n_estimators (early-stop): {best_xgb_m4.best_iteration + 1}")

results['Model 4 — Brute Force'] = evaluate_model(
    best_xgb_m4, X_train, y_train, X_test, y_test,
    label='Model 4 — Brute Force',
    features=FEATURES_M4,
    cv_score=xgb_search_m4.best_score_,
)

## Model Comparison & Best-Model Selection

All four models are scored on the same held-out test set (last 20% of the timeline). The winner is chosen by **test ROC-AUC** — the threshold-free metric that best reflects ranking quality on an imbalanced binary classification — with the train→test accuracy gap shown alongside as an overfit sanity check.

In [ ]:
# ── Build comparison frame ──────────────────────────────────────────────────
comparison_df = pd.DataFrame(list(results.values()))[
    ['model', 'n_features', 'cv_roc_auc',
     'train_accuracy', 'test_accuracy',
     'test_precision', 'test_recall', 'test_f1', 'test_roc_auc',
     'overfit_gap']
].sort_values('test_roc_auc', ascending=False).reset_index(drop=True)

print("=" * 110)
print("MODEL COMPARISON — sorted by test ROC-AUC")
print("=" * 110)
print(comparison_df.to_string(index=False))

# ── Pick the winner ─────────────────────────────────────────────────────────
winner_row   = comparison_df.iloc[0]
winner_label = winner_row['model']
winner_meta  = results[winner_label]

print("\n" + "=" * 80)
print("WINNING MODEL")
print("=" * 80)
print(f"  Model         : {winner_label}")
print(f"  # features    : {winner_row['n_features']}")
print(f"  Features      : {winner_meta['features']}")
print(f"  CV ROC-AUC    : {winner_row['cv_roc_auc']:.4f}")
print(f"  Test ROC-AUC  : {winner_row['test_roc_auc']:.4f}")
print(f"  Test accuracy : {winner_row['test_accuracy']:.4f}")
print(f"  Test F1       : {winner_row['test_f1']:.4f}")
print(f"  Overfit gap   : {winner_row['overfit_gap']:+.4f}")

# ── Visual comparison ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

x = np.arange(len(comparison_df))
labels = comparison_df['model'].tolist()

axes[0].bar(x, comparison_df['test_roc_auc'], color='steelblue',
            edgecolor='black')
axes[0].axhline(0.5, color='grey', linestyle='--', label='Random')
axes[0].set_xticks(x); axes[0].set_xticklabels(labels, rotation=20, ha='right')
axes[0].set_ylabel('Test ROC-AUC')
axes[0].set_title('Test ROC-AUC by Model', fontweight='bold')
axes[0].set_ylim(0.45, max(0.7, comparison_df['test_roc_auc'].max() + 0.05))
axes[0].legend()

width = 0.35
axes[1].bar(x - width/2, comparison_df['train_accuracy'], width,
            label='Train', color='#9ecae1', edgecolor='black')
axes[1].bar(x + width/2, comparison_df['test_accuracy'], width,
            label='Test',  color='#3182bd', edgecolor='black')
axes[1].set_xticks(x); axes[1].set_xticklabels(labels, rotation=20, ha='right')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Train vs Test Accuracy (overfit check)', fontweight='bold')
axes[1].legend()

colors = ['#d62728' if g > 0.10 else '#2ca02c'
          for g in comparison_df['overfit_gap']]
axes[2].bar(x, comparison_df['overfit_gap'], color=colors, edgecolor='black')
axes[2].axhline(0.10, color='red', linestyle='--',
                label='0.10 overfit threshold')
axes[2].set_xticks(x); axes[2].set_xticklabels(labels, rotation=20, ha='right')
axes[2].set_ylabel('Train acc − Test acc')
axes[2].set_title('Overfitting Gap by Model', fontweight='bold')
axes[2].legend()

plt.tight_layout(); plt.show()

# ── ROC curves of every model on the same axes ──────────────────────────────
plt.figure(figsize=(8, 6))
for label, model, feats in [
    ('Model 1 — All Features',     best_xgb,    FEATURES),
    ('Model 2 — 95% Importance',   best_xgb_m2, FEATURES_M2),
    ('Model 3 — RFECV',            best_xgb_m3, FEATURES_M3),
    ('Model 4 — Brute Force',      best_xgb_m4, FEATURES_M4),
]:
    proba = model.predict_proba(X_test[feats])[:, 1]
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc_val = auc(fpr, tpr)
    style = '-' if label == winner_label else '--'
    plt.plot(fpr, tpr, style, lw=2,
             label=f'{label}  (AUC = {auc_val:.4f})')
plt.plot([0, 1], [0, 1], color='grey', lw=1, linestyle=':',
         label='Random')
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC Curves — All Models (Test Set)', fontweight='bold')
plt.legend(loc='lower right'); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print("\n" + "=" * 80)
print("FINAL RECOMMENDATION")
print("=" * 80)
print(f"→ Use {winner_label} with features: {winner_meta['features']}")
print(f"  Expected test ROC-AUC: {winner_row['test_roc_auc']:.4f}")
print("=" * 80)